In [56]:
import json
import os
import random

import pandas as pd
from tqdm import tqdm
from collections import defaultdict
from itertools import combinations
from openai import OpenAI
from dotenv import load_dotenv

from utils.edinet_api import get_doc_name
from utils.datapath import edinet_codes_path, nikkei_225_path, docs_metadata_path


In [57]:
load_dotenv()
XAI_API_KEY = os.getenv("XAI_API_KEY")

client = OpenAI(
    api_key=XAI_API_KEY,
    base_url="https://api.x.ai/v1",
)

In [58]:
df = pd.read_excel(edinet_codes_path)

with open(nikkei_225_path) as f:
    nikkei_225 = json.load(f)

with open(docs_metadata_path) as f:
    docs_metadata = json.load(f)

In [ ]:
# TODO:
# Create a dictionary of quarters and document paths - DONE
# Create combinations of all documents in a quarter or random combinations - DONE
# Create winner function - DONE
# Create output dictionary of document -> number of wins (for all combinations) - DONE
# Create a dictionary of (doc1, doc2) -> winner for ELO score (for all combinations and random combinations) - DONE

# Test with random winner, and then with Grok API

In [ ]:
# Create a dictionary of quarters and document paths - DONE

quarterly_docs = defaultdict(list)

for doc in docs_metadata:
    period_end_date = doc["periodEnd"]
    period_end_ts = pd.Timestamp(period_end_date)
    period_end_quater = f"{period_end_ts.year}-{period_end_ts.quarter}"
    directory_path = os.path.join('..', "documents", period_end_quater)

    save_name = get_doc_name(doc)
    output_path = os.path.join(directory_path, save_name)

    quarterly_docs[period_end_quater].append(output_path)

In [18]:
# Create combinations of all documents in a quarter or random combinations

quarterly_combinations = defaultdict(list)

for quarter, docs in quarterly_docs.items():
    quarterly_combinations[quarter] = list(combinations(docs, 2))
    print(quarter, len(list(combinations(docs, 2))))
    

2022-4 18528
2023-1 595
2023-2 24753
2023-3 24531
2023-4 18915


In [33]:
def generate_random_pdf_pair(quarter, iterations):
    docs = quarterly_docs[quarter]
    
    for i in range(iterations):
        selected_pdfs = random.sample(docs, 2)

        yield (
            selected_pdfs[0],
            selected_pdfs[1],
        )

In [ ]:
for quarter in quarterly_docs:
    print(quarter)
    for d1, d2 in generate_random_pdf_pair(quarter, 3):
        print(d1, d2)
        

2022-4
../documents/2022-4/E01956_コナミグループ株式会社_140_S100Q6S0.pdf ../documents/2022-4/E04148_西日本旅客鉄道株式会社_140_S100Q6LU.pdf
../documents/2022-4/E01914_株式会社村田製作所_140_S100Q5FH.pdf ../documents/2022-4/E03556_株式会社千葉銀行_140_S100Q4EN.pdf
../documents/2022-4/E00346_株式会社日清製粉グループ本社_140_S100Q349.pdf ../documents/2022-4/E01486_株式会社アマダ_140_S100Q4YP.pdf
2023-1
../documents/2023-1/E21183_大塚ホールディングス株式会社_140_S100QQZ3.pdf ../documents/2023-1/E02168_ヤマハ発動機株式会社_140_S100QRB1.pdf
../documents/2023-1/E00395_キリンホールディングス株式会社_140_S100QPV6.pdf ../documents/2023-1/E00883_花王株式会社_140_S100QPZV.pdf
../documents/2023-1/E03248_株式会社　良品計画_140_S100QL68.pdf ../documents/2023-1/E02168_ヤマハ発動機株式会社_140_S100QRB1.pdf
2023-2
../documents/2023-2/E24050_ＥＮＥＯＳホールディングス株式会社_140_S100RKXL.pdf ../documents/2023-2/E05725_株式会社ＺＯＺＯ_140_S100RPHH.pdf
../documents/2023-2/E03614_株式会社三井住友フィナンシャルグループ_140_S100RP3B.pdf ../documents/2023-2/E01542_株式会社荏原製作所_140_S100ROLD.pdf
../documents/2023-2/E00023_住友金属鉱山株式会社_140_S100ROCX.pdf ../documents/2023-2/E04273_

In [76]:
from utils.llm_api import call_grok_api, prepare_prompt_messages
from utils.pdf import extract_pdf_text


def get_winner(pdf1_path, pdf2_path):
    return random.choice([1, 2])

def get_winner_grok(pdf1_path, pdf2_path):
    pdf1_txt, pdf2_txt = extract_pdf_text(pdf1_path), extract_pdf_text(pdf2_path)
    completion = call_grok_api(client, prepare_prompt_messages(pdf1_txt, pdf2_txt))
    response = json.loads(completion.choices[0].message.content)
    return response["winner"]

def get_code_from_path(pdf_path):
    """
    path = f"../documents/{quarter}/{edinet_code}_{filer}_{doc_type_code}_{doc_id}.{FILE_EXT}"
    """
    return pdf_path.split("/")[-1].split("_")[0]

In [78]:
# Test get_code_from_path()
for quarter, combinations in quarterly_combinations.items():
    for pdf1_path, pdf2_path in combinations:
        codes = [get_code_from_path(path) for path in (pdf1_path, pdf2_path)]
        print(codes)
        break

['E01741', 'E03248']
['E03248', 'E03217']
['E00143', 'E01741']
['E00143', 'E01741']
['E00143', 'E03516']


In [79]:
# Document winners
quarterly_wins = defaultdict(lambda: defaultdict(int))

for quarter, combinations in quarterly_combinations.items():
    for pdf1_path, pdf2_path in combinations:
        winner = get_winner(pdf1_path, pdf2_path)
        codes = [get_code_from_path(path) for path in (pdf1_path, pdf2_path)]
        winner_name = codes[winner - 1]
        quarterly_wins[quarter][winner_name] += 1 

In [80]:
# Battle outcomes
quarterly_battle_outcomes = defaultdict(list)

for quarter in quarterly_docs:
    for pdf1_path, pdf2_path in generate_random_pdf_pair(quarter, 2000):
        winner = get_winner(pdf1_path, pdf2_path)
        codes = [get_code_from_path(path) for path in (pdf1_path, pdf2_path)]
        quarterly_battle_outcomes[quarter].append(tuple(codes + [winner]))


In [81]:
quarterly_battle_outcomes

defaultdict(list,
            {'2022-4': [('E01741', 'E04502', 1),
              ('E31748', 'E03606', 2),
              ('E00436', 'E01873', 1),
              ('E00752', 'E01130', 1),
              ('E00988', 'E02144', 1),
              ('E00446', 'E04520', 1),
              ('E01232', 'E37777', 2),
              ('E03814', 'E37777', 1),
              ('E01231', 'E02163', 1),
              ('E05725', 'E04430', 1),
              ('E02497', 'E02367', 2),
              ('E02513', 'E00048', 1),
              ('E00767', 'E01002', 2),
              ('E01481', 'E03907', 2),
              ('E00090', 'E00436', 2),
              ('E01481', 'E03611', 2),
              ('E01124', 'E01873', 2),
              ('E03856', 'E01334', 1),
              ('E03614', 'E01892', 1),
              ('E03752', 'E01506', 2),
              ('E05072', 'E04707', 1),
              ('E02127', 'E02481', 2),
              ('E00988', 'E03847', 2),
              ('E02367', 'E01772', 2),
              ('E03611', 'E04148', 1